In [1]:
%pip install -r ../../requirements-offline.txt --quiet

/Users/utarn/projects/ml-metrology/.claude/worktrees/issue-27/.venv/bin/python: No module named pip


Note: you may need to restart the kernel to use updated packages.


# Solution — Module M2: Thai OCR bake-off

เฉลยครบทุกข้อของ `exercise.ipynb` (ผู้สอนเปิดเผยหลังจบแล็บ) — ชุดข้อมูลเดียวกัน
กับ book (20 ภาพ, seed 42)


In [2]:
import csv  # noqa: E402
from pathlib import Path  # noqa: E402


def repo_root() -> Path:
    """หา repo root โดยไล่ขึ้นจาก cwd."""
    for p in (Path.cwd(), *Path.cwd().parents):
        if (p / "requirements-offline.txt").exists():
            return p
    raise SystemExit(
        "ไม่พบ repo root — เปิด JupyterLab จาก repo root (uv run jupyter lab)"
    )


ROOT = repo_root()
DATA = ROOT / "datasets" / "thai_ocr_evaluation" / "test"
REPORTS = ROOT / "datasets" / "measurement_reports"
TESSDATA = ROOT / "datasets" / "tessdata"
assert (DATA / "metadata.csv").exists(), (
    "ไม่พบ thai_ocr_evaluation — ดู datasets/README.md"
)
assert TESSDATA.exists(), "ไม่พบ datasets/tessdata — เช็คว่า repo ครบ"
print("repo root:", ROOT)


repo root: /Users/utarn/projects/ml-metrology/.claude/worktrees/issue-27


In [3]:
import numpy as np  # noqa: E402
import pandas as pd  # noqa: E402
import jiwer  # noqa: E402, F401

meta = list(csv.DictReader(open(DATA / "metadata.csv", encoding="utf-8")))
truth = {r["file_name"]: r["text"] for r in meta}

cats: dict[str, list[str]] = {}
for r in meta:
    cats.setdefault(r["category"], []).append(r["file_name"])

rng = np.random.default_rng(42)
subset: list[tuple[str, str]] = []
for cat in sorted(cats):
    files = sorted(cats[cat])
    idx = sorted(rng.choice(len(files), 4, replace=False))
    subset += [(cat, files[i]) for i in idx]
refs = [truth[fn].strip() for _, fn in subset]
print(len(subset), "ภาพ")


20 ภาพ


## ข้อ 1 — Tesseract `tha+eng`


In [4]:
import pytesseract  # noqa: E402


def ocr_tesseract(path, lang="tha+eng") -> str:
    """OCR ไฟล์ด้วย tesseract + traineddata ใน repo."""
    cfg = f"--tessdata-dir {TESSDATA}"
    txt = pytesseract.image_to_string(str(path), lang=lang, config=cfg)
    return " ".join(txt.split())


tess2_out = {f: ocr_tesseract(DATA / f) for _, f in subset}
tess2_cer = jiwer.cer(refs, [tess2_out[f] for _, f in subset])
print(f"tha+eng CER = {tess2_cer:.3f}  (book: tha เดี่ยว = 0.723)")

enth = [(c, f) for c, f in subset if c == "document_enth"]
refs_e = [truth[f].strip() for _, f in enth]
hyps_e = [tess2_out[f] for _, f in enth]
print("document_enth CER:", round(jiwer.cer(refs_e, hyps_e), 3))


tha+eng CER = 0.381  (book: tha เดี่ยว = 0.723)
document_enth CER: 0.1


**คำตอบข้อ 1:** `tha+eng` ลด CER รวมจาก ~0.72 เหลือ ~0.38 (เกือบครึ่ง) — ช่วยทุก
ภาพที่มีตัวเลข/อังกฤษเจือปน (document_enth ดีขึ้นชัดที่สุด: ~0.1) เพราะ tesseract
ต้องมีโมเดลภาษาอังกฤษร่วมด้วยจึงอ่านส่วนละตินได้ แต่ CER รวมยังสูงกว่า EasyOCR/
thai-trocr มาก — การเติมภาษาช่วย "เฉพาะจุด" ไม่ได้แก้ engine ที่อ่านไทยไม่ได้พื้นฐาน

## ข้อ 2 — EasyOCR แยกตามหมวด


In [5]:
import easyocr  # noqa: E402
from PIL import Image  # noqa: E402

reader = easyocr.Reader(["th", "en"], gpu=False, verbose=False)

easy_out = {}
for cat, fn in subset:
    img = Image.open(DATA / fn).convert("RGB")
    lines = reader.readtext(np.array(img), detail=0)
    easy_out[fn] = " ".join(lines)

rows = [
    {"category": cat, "cer": jiwer.cer(truth[fn].strip(), easy_out[fn])}
    for cat, fn in subset
]
easy_by_cat = pd.DataFrame(rows).groupby("category")["cer"].mean()
print(easy_by_cat.round(3))


/Users/utarn/projects/ml-metrology/.claude/worktrees/issue-27/.venv/lib/python3.14/site-packages/torch/ao/nn/quantized/dynamic/modules/rnn.py:162: UserWarning: torch.quantize_per_tensor, torch.quantize_per_channel and other quantized tensor creation functions that produce tensors with dtype torch.quint8, torch.qint8, and torch.qint32 are deprecated and will be removed in a future PyTorch release. Please see https://github.com/pytorch/pytorch/issues/184982 for more information. (Triggered internally at /Users/runner/work/pytorch/pytorch/aten/src/ATen/quantized/Quantizer.cpp:116.)
  w_ih = torch.quantize_per_tensor(
/Users/utarn/projects/ml-metrology/.claude/worktrees/issue-27/.venv/lib/python3.14/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


category
document         0.153
document_enth    0.396
handwritten      0.333
real_document    0.727
scene_text       0.359
Name: cer, dtype: float64


**คำตอบข้อ 2:** หมวดที่ยากที่สุดคือ `real_document` (~0.73 — เอกสารจริงคุณภาพ
สแกน/ฟอนต์/เงาสั้นหลากหลาย) ตามด้วย `document_enth` / `scene_text`
(~0.36–0.40) และง่ายสุดคือ `document` (พิมพ์ชัด) ที่น่าสนใจคือ `handwritten`
(~0.33) ไม่แย่ที่สุดเพราะภาพลายมือในชุดนี้เป็นข้อความสั้น ๆ สไตล์เดียวกัน —
บทเรียนคือ "ยาก-ง่าย" ต้องวัดเป็น CER ต่อหมวดจริง ไม่ใช่เดาจากความรู้สึก

## ข้อ 3 — Normalize `กํา` → `กำ`


In [6]:
tess1_out = {f: ocr_tesseract(DATA / f, lang="tha") for _, f in subset}
cer_before = jiwer.cer(refs, [tess1_out[f] for _, f in subset])
print("CER ก่อน normalize:", round(cer_before, 3))


def norm(text: str) -> str:
    """แทน nikhahit+สระอา ที่แยกกันด้วยสระอำรวมตัว + strip ช่องว่าง."""
    return text.replace("ํา", "ำ").strip()


cer_after = jiwer.cer(
    refs, [norm(tess1_out[f]) for _, f in subset]
)
print("CER หลัง normalize:", round(cer_after, 3))


CER ก่อน normalize: 0.723
CER หลัง normalize: 0.715


**คำตอบข้อ 3:** CER ลดลงเล็กน้อย (~0.01 บนชุดนี้ — ภาพที่มีคำ "กำ"/สระอำมีไม่กี่ภาพ)
ประเด็นสำคัญ: **ตัวเลข CER ขึ้นกับนิยาม/การเตรียมข้อความก่อนวัดด้วย** —
ก่อนเทียบโมเดล ต้อง fix วิธี normalize ให้เหมือนกันทุกโมเดล (เช่นเดียวกับการ
สอบเทียบเครื่องมือ: ต้องระบุเงื่อนไขการวัดให้ตรงกันก่อนจึงเทียบผลกันได้)

## ข้อ 4 — Surya layout หน้าสรุปผล/ลายเซ็น


In [7]:
from surya.layout import LayoutPredictor  # noqa: E402

layout_pred = LayoutPredictor()

page = Image.open(sorted(REPORTS.glob("*.png"))[-1]).convert("RGB")
layouts = layout_pred([page])[0]
print(pd.Series([b.label for b in layouts.bboxes]).value_counts())


/Users/utarn/projects/ml-metrology/.claude/worktrees/issue-27/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Recognizing layout:   0%|          | 0/1 [00:00<?, ?it/s]

Recognizing layout: 100%|██████████| 1/1 [00:00<00:00,  1.74it/s]

Recognizing layout: 100%|██████████| 1/1 [00:00<00:00,  1.74it/s]

Text             4
PageHeader       2
SectionHeader    1
Table            1
PageFooter       1
Name: count, dtype: int64


**คำตอบข้อ 4:** ตัวอย่างผลจริง (อาจต่างเล็กน้อยระหว่างเครื่อง): `Text` 4 กล่อง
(ย่อหน้าสรุปผล + disclaimer), `PageHeader` 2, `SectionHeader` 1 (หัวข้อ
"สรุปผลการประเมิน"), `Table` 1 (บล็อกลายเซ็นถูกจับเป็นตาราง — โมเดล layout
มอง "ช่องกรอก + บรรทัด" ว่าเป็นโครงตาราง) และ `PageFooter` 1 — จำนวนกล่อง
ไม่ตายตัว แต่ label set ควรใกล้เคียงนี้ และจะเห็นว่า label เป็น "การจับของ
โมเดล" ไม่ใช่ความจริงสัมบูรณ์ (ลายเซ็น ≠ ตาราง) ต้องตรวจก่อนใช้

## สรุป

- โมเดลที่ CER ดีที่สุด: **thai-trocr** (~0.1) — ตามด้วย EasyOCR (~0.3)
- CER "จริง" ต่างได้จากที่วัด: normalize ข้อความ, ชุดภาพที่เลือก, ภาพหลายบรรทัด
  (trocr เป็น line-level), เกณฑ์การใช้งานจริง (ตัวเลขค่าการวัดผิดพลาดแพงกว่าตัวอักษรทั่วไป)
- Capstone Track C: Surya layout/table แยกโครงสร้าง → OCR ต่อช่อง → ตรวจตัวเลข
  ด้วยกฎของหน่วยงาน/การตรวจซ้ำ
